# Fase 6 — consolidação e relatório final
Esta etapa é CPU-only: reutiliza os JSONL do Drive e não carrega modelos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, subprocess

PROJECT = Path('/content/cot-enem')
DRIVE_ROOT = Path('/content/drive/MyDrive/cot-enem')
REPOSITORY_URL = 'https://github.com/cidadaofred/cot-enem.git'
BRANCH = 'master'

os.chdir('/content')
if not PROJECT.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH,
        REPOSITORY_URL, str(PROJECT),
    ], check=True)
elif not (PROJECT / '.git').is_dir():
    raise RuntimeError(f'{PROJECT} existe, mas não é um clone Git válido')
else:
    subprocess.run([
        'git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH,
    ], check=True)

os.chdir(PROJECT)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

In [ ]:
required_inputs = [
    DRIVE_ROOT / 'data/processed/enem_normalized.jsonl',
    DRIVE_ROOT / 'outputs/datasets/cot_enem_specify_ensemble_v1.jsonl',
    DRIVE_ROOT / 'outputs/datasets/complicate/results.jsonl',
    DRIVE_ROOT / 'outputs/datasets/diversify/results.jsonl',
]
missing = [str(path) for path in required_inputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Arquivos de entrada ausentes no Drive:\n- ' + '\n- '.join(missing))

command = [
    'python', '-m', 'cot_enem.cli', 'finalize',
    '--normalized', str(DRIVE_ROOT / 'data/processed/enem_normalized.jsonl'),
    '--specify', str(DRIVE_ROOT / 'outputs/datasets/cot_enem_specify_ensemble_v1.jsonl'),
    '--complicate', str(DRIVE_ROOT / 'outputs/datasets/complicate/results.jsonl'),
    '--diversify', str(DRIVE_ROOT / 'outputs/datasets/diversify/results.jsonl'),
    '--output-dir', str(DRIVE_ROOT / 'outputs/final'),
]
subprocess.run(command, check=True)

In [ ]:
FINAL = DRIVE_ROOT / 'outputs/final'
metrics = json.loads((FINAL / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('\n' + (FINAL / 'REPORT.md').read_text(encoding='utf-8'))